In [2]:
import numpy as np
import cupy as cp
import torch
import clip
from tqdm.notebook import tqdm
from pkg_resources import packaging
from imagenetv2_pytorch import ImageNetV2Dataset

print("Torch version:", torch.__version__)

with torch.no_grad():
    torch.cuda.empty_cache()

#pip install git+https://github.com/openai/CLIP.git
#pip install cupy-cuda11x
#pip install git+https://github.com/modestyachts/ImageNetV2_pytorch

Torch version: 2.0.0+cu117


In [3]:
# def clip_eval(model, dataloader, imagenet_classes, imagenet_templates, device):
#     zeroshot_weights = zeroshot_classifier(imagenet_classes, imagenet_templates, device)

#     top1, top5 = clip_eval_imagenetv2(model, dataloader, zeroshot_weights, device)

#     return top1, top5



def clip_eval_imagenetv2(model, loader, zeroshot_weights, device):
    with torch.no_grad():
        top1, top5, n = 0., 0., 0.
        for i, (images, target) in enumerate(tqdm(loader, disable = True)):
            images = images.to(device)
            target = target.to(device)
            # predict
            image_features = model.encode_image(images)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            logits = 100. * image_features @ zeroshot_weights
            # measure accuracy
            acc1, acc5 = accuracy(logits, target, topk=(1, 5))
            top1 += acc1
            top5 += acc5
            n += images.size(0)
    top1 = (top1 / n) * 100
    top5 = (top5 / n) * 100

    return top1, top5

def accuracy(output, target, topk=(1,)):
    pred = output.topk(max(topk), 1, True, True)[1].t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    return [float(correct[:k].reshape(-1).float().sum(0, keepdim=True).cpu().numpy()) for k in topk]

def build_clip_idx(model):
    quantizable_idx = {'all': [], 'text': [], 'vision': []}
    # from clip model
    vision_idx = 4
    text_idx = 115

    trainable_layer_types = []

    for name, module in model.named_modules():
        if hasattr(module, 'weight') and module.weight.requires_grad:
            layer_type = type(module)
            if layer_type not in trainable_layer_types:
                trainable_layer_types.append(layer_type)

    for i, module in enumerate(model.modules()):
        if type(module) in trainable_layer_types:
            if (i < text_idx):
                quantizable_idx['vision'].append(i)
            if (i > text_idx):
                quantizable_idx['text'].append(i)
    quantizable_idx['all'] = quantizable_idx['vision'] + quantizable_idx['text']
    #quantizable_idx['all'] = quantizable_idx['text']
    return quantizable_idx

def eval_noisy_model(model_name,
                     imagenet_classes,
                     imagenet_templates,
                     dataloader,
                     layer_idx,
                     epoch_num,
                     mask,
                     noise,
                     prob,
                     code,
                     device):
    acc1_arr = []
    acc5_arr = []
    for i in range(epoch_num):
        model, preprocess = clip.load(model_name)

        add_noise_float_model(model,mask,noise,prob,layer_idx,device,code=code)
        
        zeroshot_weights = []
        with torch.no_grad():
            for classname in tqdm(imagenet_classes, disable = True):
                texts = [template.format(classname) for template in imagenet_templates] #format with class
                texts = clip.tokenize(texts).cuda() #tokenize
                class_embeddings = model.encode_text(texts) #embed with text encoder
                class_embeddings /= class_embeddings.norm(dim=-1, keepdim=True)
                class_embedding = class_embeddings.mean(dim=0)
                class_embedding /= class_embedding.norm()
                zeroshot_weights.append(class_embedding)
            zeroshot_weights = torch.stack(zeroshot_weights, dim=1).to(device)
        
        top1, top5 = clip_eval_imagenetv2(
            model,
            dataloader,
            zeroshot_weights,
            device)
        # print('idx')
        # print(i)
        # print(acc)
        acc1_arr.append(top1)
        acc5_arr.append(top5)
    print('prob: {}'.format(prob))
    print('accuracy list: {}'.format(acc1_arr))
    print('accuracy list: {}'.format(acc5_arr))
    acc1_total = sum(acc1_arr) / epoch_num
    acc5_total = sum(acc5_arr) / epoch_num
    print('average accuracy: {}'.format(acc1_total))
    print('average accuracy: {}'.format(acc5_total))
    #print()
    return acc1_arr, acc1_total, acc5_arr, acc5_total

def add_noise_float_model(model, important_mask, noise, prob, mask_index, device, bitwidth=32, is_pruned=False, code=None):
    layer_mask_dict = {n: m for (n, m) in zip(mask_index, important_mask)}
    for i, layer in enumerate(model.modules()):
        if i not in mask_index:
            continue
        mask = layer_mask_dict[i]
        if i > 115:
            prob = 1e-6
        else:
            prob = 1e-2
        if code == None:
            if(layer.weight.data.dtype == torch.float16):
                noisy_weight_data = add_noise_float_layer_16(layer.weight.data, mask, noise, prob, 16)
            else:
                noisy_weight_data = add_noise_float_layer_32(layer.weight.data, mask, noise, prob, 32)
        layer.weight.data = noisy_weight_data.to(device)

def add_noise_float_layer_32(data, mask, noise, prob, bitwidth):
    mask = cp.asarray(mask, dtype=cp.uint32)
    # mask_2 = cp.asarray(mask_2, dtype=cp.uint32)
    data = cp.asarray(data.cpu(), dtype=cp.float32)
    data_shape = data.shape

    data = data.reshape(-1)
    float_to_int = cp.RawKernel(r'''
        extern "C" __global__
        void float_to_int(const float * x, unsigned int * y) {
            int tid = blockDim.x * blockIdx.x + threadIdx.x;
            memcpy(&(y[tid]), &(x[tid]), sizeof(float));
        }
    ''', 'float_to_int')
    x_p = cp.zeros(data.shape, dtype=cp.uint32)
    float_to_int(data.shape, (1, ), (data, x_p))

    noise_mask_arr = cp.random.rand(data.size, bitwidth)
    noise_mask_arr = (noise_mask_arr < prob).astype(cp.uint32)
    noise_mask = cp.sum(noise_mask_arr * (2 ** cp.arange(bitwidth-1,-1,-1)), axis=1)
    noise_mask = noise_mask.astype(cp.uint32)

    # x_p_2 = x_p ^ (noise_mask & (~mask_2))
    x_p = x_p ^ (noise_mask & (~mask))

    int_to_float = cp.RawKernel(r'''
        extern "C" __global__
        void int_to_float(const unsigned int * x, float * y) {
            int tid = blockDim.x * blockIdx.x + threadIdx.x;
            memcpy(&(y[tid]), &(x[tid]), sizeof(float));
        }
    ''', 'int_to_float')
    output = cp.zeros(data.shape, dtype=cp.float32)
    int_to_float(data.shape, (1, ), (x_p, output))
    output = output.reshape(data_shape)
    output = cp.asnumpy(output)
    output = torch.as_tensor(output, dtype=torch.float32)

    return output

def add_noise_float_layer_16(data, mask, noise, prob, bitwidth):
    mask = cp.asarray(mask, dtype=cp.uint32)
    # mask_2 = cp.asarray(mask_2, dtype=cp.uint32)
    data = cp.asarray(data.cpu(), dtype=cp.float32)
    data_shape = data.shape

    data = data.reshape(-1)
    float_to_int = cp.RawKernel(r'''
        extern "C" __global__
        void float_to_int(const float * x, unsigned int * y) {
            int tid = blockDim.x * blockIdx.x + threadIdx.x;
            memcpy(&(y[tid]), &(x[tid]), sizeof(float));
        }
    ''', 'float_to_int')
    x_p = cp.zeros(data.shape, dtype=cp.uint32)
    float_to_int(data.shape, (1, ), (data, x_p))

    noise_mask_arr = cp.random.rand(data.size, bitwidth)
    noise_mask_arr = (noise_mask_arr < prob).astype(cp.uint32)
    noise_mask = cp.sum(noise_mask_arr * (2 ** cp.arange(bitwidth-1,-1,-1)), axis=1)
    noise_mask = noise_mask.astype(cp.uint32)

    # x_p_2 = x_p ^ (noise_mask & (~mask_2))
    x_p = x_p ^ (noise_mask & (~mask))

    int_to_float = cp.RawKernel(r'''
        extern "C" __global__
        void int_to_float(const unsigned int * x, float * y) {
            int tid = blockDim.x * blockIdx.x + threadIdx.x;
            memcpy(&(y[tid]), &(x[tid]), sizeof(float));
        }
    ''', 'int_to_float')
    output = cp.zeros(data.shape, dtype=cp.float32)
    int_to_float(data.shape, (1, ), (x_p, output))
    output = output.reshape(data_shape)
    output = cp.asnumpy(output)
    output = torch.as_tensor(output, dtype=torch.float16)

    return output

def convert_mask(mask):
    mask_result = []
    for m in mask:
        m_result = 0
        for bit in m:
            m_result <<= 1
            m_result += int(bit)
        mask_result.append(m_result)
    return mask_result

# def convert_mask(mask):
#     return [int(''.join(bit), 2) for bit in mask]

In [4]:
imagenet_classes = ["tench", "goldfish", "great white shark", "tiger shark", "hammerhead shark", "electric ray", "stingray", "rooster", "hen", "ostrich", "brambling", "goldfinch", "house finch", "junco", "indigo bunting", "American robin", "bulbul", "jay", "magpie", "chickadee", "American dipper", "kite (bird of prey)", "bald eagle", "vulture", "great grey owl", "fire salamander", "smooth newt", "newt", "spotted salamander", "axolotl", "American bullfrog", "tree frog", "tailed frog", "loggerhead sea turtle", "leatherback sea turtle", "mud turtle", "terrapin", "box turtle", "banded gecko", "green iguana", "Carolina anole", "desert grassland whiptail lizard", "agama", "frilled-necked lizard", "alligator lizard", "Gila monster", "European green lizard", "chameleon", "Komodo dragon", "Nile crocodile", "American alligator", "triceratops", "worm snake", "ring-necked snake", "eastern hog-nosed snake", "smooth green snake", "kingsnake", "garter snake", "water snake", "vine snake", "night snake", "boa constrictor", "African rock python", "Indian cobra", "green mamba", "sea snake", "Saharan horned viper", "eastern diamondback rattlesnake", "sidewinder rattlesnake", "trilobite", "harvestman", "scorpion", "yellow garden spider", "barn spider", "European garden spider", "southern black widow", "tarantula", "wolf spider", "tick", "centipede", "black grouse", "ptarmigan", "ruffed grouse", "prairie grouse", "peafowl", "quail", "partridge", "african grey parrot", "macaw", "sulphur-crested cockatoo", "lorikeet", "coucal", "bee eater", "hornbill", "hummingbird", "jacamar", "toucan", "duck", "red-breasted merganser", "goose", "black swan", "tusker", "echidna", "platypus", "wallaby", "koala", "wombat", "jellyfish", "sea anemone", "brain coral", "flatworm", "nematode", "conch", "snail", "slug", "sea slug", "chiton", "chambered nautilus", "Dungeness crab", "rock crab", "fiddler crab", "red king crab", "American lobster", "spiny lobster", "crayfish", "hermit crab", "isopod", "white stork", "black stork", "spoonbill", "flamingo", "little blue heron", "great egret", "bittern bird", "crane bird", "limpkin", "common gallinule", "American coot", "bustard", "ruddy turnstone", "dunlin", "common redshank", "dowitcher", "oystercatcher", "pelican", "king penguin", "albatross", "grey whale", "killer whale", "dugong", "sea lion", "Chihuahua", "Japanese Chin", "Maltese", "Pekingese", "Shih Tzu", "King Charles Spaniel", "Papillon", "toy terrier", "Rhodesian Ridgeback", "Afghan Hound", "Basset Hound", "Beagle", "Bloodhound", "Bluetick Coonhound", "Black and Tan Coonhound", "Treeing Walker Coonhound", "English foxhound", "Redbone Coonhound", "borzoi", "Irish Wolfhound", "Italian Greyhound", "Whippet", "Ibizan Hound", "Norwegian Elkhound", "Otterhound", "Saluki", "Scottish Deerhound", "Weimaraner", "Staffordshire Bull Terrier", "American Staffordshire Terrier", "Bedlington Terrier", "Border Terrier", "Kerry Blue Terrier", "Irish Terrier", "Norfolk Terrier", "Norwich Terrier", "Yorkshire Terrier", "Wire Fox Terrier", "Lakeland Terrier", "Sealyham Terrier", "Airedale Terrier", "Cairn Terrier", "Australian Terrier", "Dandie Dinmont Terrier", "Boston Terrier", "Miniature Schnauzer", "Giant Schnauzer", "Standard Schnauzer", "Scottish Terrier", "Tibetan Terrier", "Australian Silky Terrier", "Soft-coated Wheaten Terrier", "West Highland White Terrier", "Lhasa Apso", "Flat-Coated Retriever", "Curly-coated Retriever", "Golden Retriever", "Labrador Retriever", "Chesapeake Bay Retriever", "German Shorthaired Pointer", "Vizsla", "English Setter", "Irish Setter", "Gordon Setter", "Brittany dog", "Clumber Spaniel", "English Springer Spaniel", "Welsh Springer Spaniel", "Cocker Spaniel", "Sussex Spaniel", "Irish Water Spaniel", "Kuvasz", "Schipperke", "Groenendael dog", "Malinois", "Briard", "Australian Kelpie", "Komondor", "Old English Sheepdog", "Shetland Sheepdog", "collie", "Border Collie", "Bouvier des Flandres dog", "Rottweiler", "German Shepherd Dog", "Dobermann", "Miniature Pinscher", "Greater Swiss Mountain Dog", "Bernese Mountain Dog", "Appenzeller Sennenhund", "Entlebucher Sennenhund", "Boxer", "Bullmastiff", "Tibetan Mastiff", "French Bulldog", "Great Dane", "St. Bernard", "husky", "Alaskan Malamute", "Siberian Husky", "Dalmatian", "Affenpinscher", "Basenji", "pug", "Leonberger", "Newfoundland dog", "Great Pyrenees dog", "Samoyed", "Pomeranian", "Chow Chow", "Keeshond", "brussels griffon", "Pembroke Welsh Corgi", "Cardigan Welsh Corgi", "Toy Poodle", "Miniature Poodle", "Standard Poodle", "Mexican hairless dog (xoloitzcuintli)", "grey wolf", "Alaskan tundra wolf", "red wolf or maned wolf", "coyote", "dingo", "dhole", "African wild dog", "hyena", "red fox", "kit fox", "Arctic fox", "grey fox", "tabby cat", "tiger cat", "Persian cat", "Siamese cat", "Egyptian Mau", "cougar", "lynx", "leopard", "snow leopard", "jaguar", "lion", "tiger", "cheetah", "brown bear", "American black bear", "polar bear", "sloth bear", "mongoose", "meerkat", "tiger beetle", "ladybug", "ground beetle", "longhorn beetle", "leaf beetle", "dung beetle", "rhinoceros beetle", "weevil", "fly", "bee", "ant", "grasshopper", "cricket insect", "stick insect", "cockroach", "praying mantis", "cicada", "leafhopper", "lacewing", "dragonfly", "damselfly", "red admiral butterfly", "ringlet butterfly", "monarch butterfly", "small white butterfly", "sulphur butterfly", "gossamer-winged butterfly", "starfish", "sea urchin", "sea cucumber", "cottontail rabbit", "hare", "Angora rabbit", "hamster", "porcupine", "fox squirrel", "marmot", "beaver", "guinea pig", "common sorrel horse", "zebra", "pig", "wild boar", "warthog", "hippopotamus", "ox", "water buffalo", "bison", "ram (adult male sheep)", "bighorn sheep", "Alpine ibex", "hartebeest", "impala (antelope)", "gazelle", "arabian camel", "llama", "weasel", "mink", "European polecat", "black-footed ferret", "otter", "skunk", "badger", "armadillo", "three-toed sloth", "orangutan", "gorilla", "chimpanzee", "gibbon", "siamang", "guenon", "patas monkey", "baboon", "macaque", "langur", "black-and-white colobus", "proboscis monkey", "marmoset", "white-headed capuchin", "howler monkey", "titi monkey", "Geoffroy's spider monkey", "common squirrel monkey", "ring-tailed lemur", "indri", "Asian elephant", "African bush elephant", "red panda", "giant panda", "snoek fish", "eel", "silver salmon", "rock beauty fish", "clownfish", "sturgeon", "gar fish", "lionfish", "pufferfish", "abacus", "abaya", "academic gown", "accordion", "acoustic guitar", "aircraft carrier", "airliner", "airship", "altar", "ambulance", "amphibious vehicle", "analog clock", "apiary", "apron", "trash can", "assault rifle", "backpack", "bakery", "balance beam", "balloon", "ballpoint pen", "Band-Aid", "banjo", "baluster / handrail", "barbell", "barber chair", "barbershop", "barn", "barometer", "barrel", "wheelbarrow", "baseball", "basketball", "bassinet", "bassoon", "swimming cap", "bath towel", "bathtub", "station wagon", "lighthouse", "beaker", "military hat (bearskin or shako)", "beer bottle", "beer glass", "bell tower", "baby bib", "tandem bicycle", "bikini", "ring binder", "binoculars", "birdhouse", "boathouse", "bobsleigh", "bolo tie", "poke bonnet", "bookcase", "bookstore", "bottle cap", "hunting bow", "bow tie", "brass memorial plaque", "bra", "breakwater", "breastplate", "broom", "bucket", "buckle", "bulletproof vest", "high-speed train", "butcher shop", "taxicab", "cauldron", "candle", "cannon", "canoe", "can opener", "cardigan", "car mirror", "carousel", "tool kit", "cardboard box / carton", "car wheel", "automated teller machine", "cassette", "cassette player", "castle", "catamaran", "CD player", "cello", "mobile phone", "chain", "chain-link fence", "chain mail", "chainsaw", "storage chest", "chiffonier", "bell or wind chime", "china cabinet", "Christmas stocking", "church", "movie theater", "cleaver", "cliff dwelling", "cloak", "clogs", "cocktail shaker", "coffee mug", "coffeemaker", "spiral or coil", "combination lock", "computer keyboard", "candy store", "container ship", "convertible", "corkscrew", "cornet", "cowboy boot", "cowboy hat", "cradle", "construction crane", "crash helmet", "crate", "infant bed", "Crock Pot", "croquet ball", "crutch", "cuirass", "dam", "desk", "desktop computer", "rotary dial telephone", "diaper", "digital clock", "digital watch", "dining table", "dishcloth", "dishwasher", "disc brake", "dock", "dog sled", "dome", "doormat", "drilling rig", "drum", "drumstick", "dumbbell", "Dutch oven", "electric fan", "electric guitar", "electric locomotive", "entertainment center", "envelope", "espresso machine", "face powder", "feather boa", "filing cabinet", "fireboat", "fire truck", "fire screen", "flagpole", "flute", "folding chair", "football helmet", "forklift", "fountain", "fountain pen", "four-poster bed", "freight car", "French horn", "frying pan", "fur coat", "garbage truck", "gas mask or respirator", "gas pump", "goblet", "go-kart", "golf ball", "golf cart", "gondola", "gong", "gown", "grand piano", "greenhouse", "radiator grille", "grocery store", "guillotine", "hair clip", "hair spray", "half-track", "hammer", "hamper", "hair dryer", "hand-held computer", "handkerchief", "hard disk drive", "harmonica", "harp", "combine harvester", "hatchet", "holster", "home theater", "honeycomb", "hook", "hoop skirt", "gymnastic horizontal bar", "horse-drawn vehicle", "hourglass", "iPod", "clothes iron", "carved pumpkin", "jeans", "jeep", "T-shirt", "jigsaw puzzle", "rickshaw", "joystick", "kimono", "knee pad", "knot", "lab coat", "ladle", "lampshade", "laptop computer", "lawn mower", "lens cap", "letter opener", "library", "lifeboat", "lighter", "limousine", "ocean liner", "lipstick", "slip-on shoe", "lotion", "music speaker", "loupe magnifying glass", "sawmill", "magnetic compass", "messenger bag", "mailbox", "tights", "one-piece bathing suit", "manhole cover", "maraca", "marimba", "mask", "matchstick", "maypole", "maze", "measuring cup", "medicine cabinet", "megalith", "microphone", "microwave oven", "military uniform", "milk can", "minibus", "miniskirt", "minivan", "missile", "mitten", "mixing bowl", "mobile home", "ford model t", "modem", "monastery", "monitor", "moped", "mortar and pestle", "graduation cap", "mosque", "mosquito net", "vespa", "mountain bike", "tent", "computer mouse", "mousetrap", "moving van", "muzzle", "metal nail", "neck brace", "necklace", "baby pacifier", "notebook computer", "obelisk", "oboe", "ocarina", "odometer", "oil filter", "pipe organ", "oscilloscope", "overskirt", "bullock cart", "oxygen mask", "product packet / packaging", "paddle", "paddle wheel", "padlock", "paintbrush", "pajamas", "palace", "pan flute", "paper towel", "parachute", "parallel bars", "park bench", "parking meter", "railroad car", "patio", "payphone", "pedestal", "pencil case", "pencil sharpener", "perfume", "Petri dish", "photocopier", "plectrum", "Pickelhaube", "picket fence", "pickup truck", "pier", "piggy bank", "pill bottle", "pillow", "ping-pong ball", "pinwheel", "pirate ship", "drink pitcher", "block plane", "planetarium", "plastic bag", "plate rack", "farm plow", "plunger", "Polaroid camera", "pole", "police van", "poncho", "pool table", "soda bottle", "plant pot", "potter's wheel", "power drill", "prayer rug", "printer", "prison", "missile", "projector", "hockey puck", "punching bag", "purse", "quill", "quilt", "race car", "racket", "radiator", "radio", "radio telescope", "rain barrel", "recreational vehicle", "fishing casting reel", "reflex camera", "refrigerator", "remote control", "restaurant", "revolver", "rifle", "rocking chair", "rotisserie", "eraser", "rugby ball", "ruler measuring stick", "sneaker", "safe", "safety pin", "salt shaker", "sandal", "sarong", "saxophone", "scabbard", "weighing scale", "school bus", "schooner", "scoreboard", "CRT monitor", "screw", "screwdriver", "seat belt", "sewing machine", "shield", "shoe store", "shoji screen / room divider", "shopping basket", "shopping cart", "shovel", "shower cap", "shower curtain", "ski", "balaclava ski mask", "sleeping bag", "slide rule", "sliding door", "slot machine", "snorkel", "snowmobile", "snowplow", "soap dispenser", "soccer ball", "sock", "solar thermal collector", "sombrero", "soup bowl", "keyboard space bar", "space heater", "space shuttle", "spatula", "motorboat", "spider web", "spindle", "sports car", "spotlight", "stage", "steam locomotive", "through arch bridge", "steel drum", "stethoscope", "scarf", "stone wall", "stopwatch", "stove", "strainer", "tram", "stretcher", "couch", "stupa", "submarine", "suit", "sundial", "sunglasses", "sunglasses", "sunscreen", "suspension bridge", "mop", "sweatshirt", "swim trunks / shorts", "swing", "electrical switch", "syringe", "table lamp", "tank", "tape player", "teapot", "teddy bear", "television", "tennis ball", "thatched roof", "front curtain", "thimble", "threshing machine", "throne", "tile roof", "toaster", "tobacco shop", "toilet seat", "torch", "totem pole", "tow truck", "toy store", "tractor", "semi-trailer truck", "tray", "trench coat", "tricycle", "trimaran", "tripod", "triumphal arch", "trolleybus", "trombone", "hot tub", "turnstile", "typewriter keyboard", "umbrella", "unicycle", "upright piano", "vacuum cleaner", "vase", "vaulted or arched ceiling", "velvet fabric", "vending machine", "vestment", "viaduct", "violin", "volleyball", "waffle iron", "wall clock", "wallet", "wardrobe", "military aircraft", "sink", "washing machine", "water bottle", "water jug", "water tower", "whiskey jug", "whistle", "hair wig", "window screen", "window shade", "Windsor tie", "wine bottle", "airplane wing", "wok", "wooden spoon", "wool", "split-rail fence", "shipwreck", "sailboat", "yurt", "website", "comic book", "crossword", "traffic or street sign", "traffic light", "dust jacket", "menu", "plate", "guacamole", "consomme", "hot pot", "trifle", "ice cream", "popsicle", "baguette", "bagel", "pretzel", "cheeseburger", "hot dog", "mashed potatoes", "cabbage", "broccoli", "cauliflower", "zucchini", "spaghetti squash", "acorn squash", "butternut squash", "cucumber", "artichoke", "bell pepper", "cardoon", "mushroom", "Granny Smith apple", "strawberry", "orange", "lemon", "fig", "pineapple", "banana", "jackfruit", "cherimoya (custard apple)", "pomegranate", "hay", "carbonara", "chocolate syrup", "dough", "meatloaf", "pizza", "pot pie", "burrito", "red wine", "espresso", "tea cup", "eggnog", "mountain", "bubble", "cliff", "coral reef", "geyser", "lakeshore", "promontory", "sandbar", "beach", "valley", "volcano", "baseball player", "bridegroom", "scuba diver", "rapeseed", "daisy", "yellow lady's slipper", "corn", "acorn", "rose hip", "horse chestnut seed", "coral fungus", "agaric", "gyromitra", "stinkhorn mushroom", "earth star fungus", "hen of the woods mushroom", "bolete", "corn cob", "toilet paper"]

imagenet_templates = [
    'a bad photo of a {}.',
    'a photo of many {}.',
    'a sculpture of a {}.',
    'a photo of the hard to see {}.',
    'a low resolution photo of the {}.',
    'a rendering of a {}.',
    'graffiti of a {}.',
    'a bad photo of the {}.',
    'a cropped photo of the {}.',
    'a tattoo of a {}.',
    'the embroidered {}.',
    'a photo of a hard to see {}.',
    'a bright photo of a {}.',
    'a photo of a clean {}.',
    'a photo of a dirty {}.',
    'a dark photo of the {}.',
    'a drawing of a {}.',
    'a photo of my {}.',
    'the plastic {}.',
    'a photo of the cool {}.',
    'a close-up photo of a {}.',
    'a black and white photo of the {}.',
    'a painting of the {}.',
    'a painting of a {}.',
    'a pixelated photo of the {}.',
    'a sculpture of the {}.',
    'a bright photo of the {}.',
    'a cropped photo of a {}.',
    'a plastic {}.',
    'a photo of the dirty {}.',
    'a jpeg corrupted photo of a {}.',
    'a blurry photo of the {}.',
    'a photo of the {}.',
    'a good photo of the {}.',
    'a rendering of the {}.',
    'a {} in a video game.',
    'a photo of one {}.',
    'a doodle of a {}.',
    'a close-up photo of the {}.',
    'a photo of a {}.',
    'the origami {}.',
    'the {} in a video game.',
    'a sketch of a {}.',
    'a doodle of the {}.',
    'a origami {}.',
    'a low resolution photo of a {}.',
    'the toy {}.',
    'a rendition of the {}.',
    'a photo of the clean {}.',
    'a photo of a large {}.',
    'a rendition of a {}.',
    'a photo of a nice {}.',
    'a photo of a weird {}.',
    'a blurry photo of a {}.',
    'a cartoon {}.',
    'art of a {}.',
    'a sketch of the {}.',
    'a embroidered {}.',
    'a pixelated photo of a {}.',
    'itap of the {}.',
    'a jpeg corrupted photo of the {}.',
    'a good photo of a {}.',
    'a plushie {}.',
    'a photo of the nice {}.',
    'a photo of the small {}.',
    'a photo of the weird {}.',
    'the cartoon {}.',
    'art of the {}.',
    'a drawing of the {}.',
    'a photo of the large {}.',
    'a black and white photo of a {}.',
    'the plushie {}.',
    'a dark photo of a {}.',
    'itap of a {}.',
    'graffiti of the {}.',
    'a toy {}.',
    'itap of my {}.',
    'a photo of a cool {}.',
    'a photo of a small {}.',
    'a tattoo of the {}.',
]

print(f"{len(imagenet_classes)} classes, {len(imagenet_templates)} templates")

1000 classes, 80 templates


In [5]:
seed = 7
bit = 32
epoch_num = 10
num_workers = 16

if seed > 0:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

model_name = "ViT-B/32"
model, preprocess = clip.load("ViT-B/32",jit=False)
device = "cuda" if torch.cuda.is_available() else "cpu"

images = ImageNetV2Dataset(transform=preprocess)
dataloader = torch.utils.data.DataLoader(images, batch_size=8, num_workers=2)

layer_idx_process = build_clip_idx(model)
layer_idx = layer_idx_process['all']
#mask = ['00000000000000000000000000000000'] * len(layer_idx)
mask = ['11111111111100000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)

print(mask)

noise = 'Random'
code = None

# prob_list = [2e-10, 22e-11, 24e-11, 26e-11, 28e-11, 3e-10, 4e-10, 5e-10]
prob_list = []
for exp in range(-10, 0, 1):
    prob_list.append(1*10**(exp))
    prob_list.append(5*10**(exp))
result_arr = {}
result_sum = {}

#prob_list =[1e-6,2e-6,3e-6,4e-6,5e-6,6e-6,7e-6,8e-6,9e-6,1e-5,2e-5,3e-5,4e-5,5e-5,6e-5,7e-5,8e-5,9e-5,1e-4]
#prob_list =[1e-7,2e-7,3e-7,4e-7,5e-7,6e-7,7e-7,8e-7,9e-7,1e-6]
print(prob_list)

[4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 4293918720, 429

In [6]:
#layer_idx = layer_idx[61:]
print(layer_idx)
print(len(layer_idx))

[2, 3, 8, 9, 11, 13, 14, 17, 18, 20, 22, 23, 26, 27, 29, 31, 32, 35, 36, 38, 40, 41, 44, 45, 47, 49, 50, 53, 54, 56, 58, 59, 62, 63, 65, 67, 68, 71, 72, 74, 76, 77, 80, 81, 83, 85, 86, 89, 90, 92, 94, 95, 98, 99, 101, 103, 104, 107, 108, 110, 112, 113, 114, 119, 120, 122, 124, 125, 128, 129, 131, 133, 134, 137, 138, 140, 142, 143, 146, 147, 149, 151, 152, 155, 156, 158, 160, 161, 164, 165, 167, 169, 170, 173, 174, 176, 178, 179, 182, 183, 185, 187, 188, 191, 192, 194, 196, 197, 200, 201, 203, 205, 206, 209, 210, 212, 214, 215, 218, 219, 221, 223, 224, 225, 226]
125


In [7]:
mask_list = []

mask = ['11100000000000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11110000000000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111000000000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111100000000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111110000000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111111000000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111111100000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111111110000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)

mask = ['11111111111000000000000000000000'] * len(layer_idx)
mask = convert_mask(mask)
mask_list.append(mask)


In [8]:
for mask in mask_list:
        print('mask')
        print(mask)
        prob = 1e-2
        acc1_arr, acc1_sum, acc5_arr, acc5_sum = eval_noisy_model(
            model_name,
            imagenet_classes,
            imagenet_templates,
            dataloader,
            layer_idx,
            epoch_num,
            mask,
            noise,
            prob,
            code,
            device)
        result_arr[prob] = {'1': acc1_arr, '5': acc5_arr}
        result_sum[prob] = {'1': acc1_sum, '5': acc5_sum}

mask
[3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384, 3758096384

/tmp/ipykernel_3988657/2612585549.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return [float(correct[:k].reshape(-1).float().sum(0, keepdim=True).cpu().numpy()) for k in topk]


prob: 0.01
accuracy list: [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
accuracy list: [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
average accuracy: 0.09999999999999999
average accuracy: 0.5
mask
[4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840, 4026531840,